# Data cleaning and normalizing scripts

## IEC-62443

In [ ]:
import pandas as pd
import json

# Definição dos caminhos dos arquivos com base na estrutura do repositório
arquivos_origem = [
    "../data/input/iec62443/SLES12-DISA-STIG.csv", 
    "../data/input/iec62443/SLES15-DISA-STIG.csv", 
    "../data/input/iec62443/SLES15-PCI-DSS.csv"    
]

def gerar_dataset_iec62443_limpo():
    """
    Lê os arquivos do SUSE, extrai normas IEC 62443 e descrições,
    normaliza as tags SR e exporta um dicionário consolidado para JSON.
    """
    dataframes_processados = []
    
    # Função para tratar tags mal formatadas (ex: 'SR,1.1,SR,1.2' -> ['SR 1.1', 'SR 1.2'])
    def parse_tags_sr(valor_bruto):
        if pd.isna(valor_bruto) or str(valor_bruto).strip() == "":
            return []
        partes = str(valor_bruto).split(',')
        # Agrupa os tokens em pares para formar a tag completa (ex: 'SR' + '1.1')
        return [' '.join(partes[i:i+2]) for i in range(0, len(partes), 2)]

    for arquivo in arquivos_origem:
        try:
            # Leitura com delimitador ';' e encoding windows-1252 conforme metadados
            df_temp = pd.read_csv(arquivo, sep=';', encoding='windows-1252')
            
            # Extraímos a norma (IEC.62443) e a descrição técnica (coluna 'Name')
            # Nota: 'Name' contém o racional técnico da configuração, ideal para o mapeamento
            dataframes_processados.append(df_temp[["IEC.62443", "Name"]])
        except Exception as e:
            print(f"Erro ao processar {arquivo}: {e}")

    # 1. União e Limpeza Inicial
    df_consolidado = pd.concat(dataframes_processados, ignore_index=True)
    df_consolidado = df_consolidado.dropna(subset=["IEC.62443"])
    
    # 2. Normalização das Normas (Explode)
    # Aplicamos o split para transformar strings em listas e 'explodimos' o dataframe
    df_consolidado["IEC.62443"] = df_consolidado["IEC.62443"].apply(parse_tags_sr)
    df_normalizado = df_consolidado.explode("IEC.62443")
    
    # Removemos entradas vazias resultantes de espaços ou nulos e duplicatas exatas
    df_normalizado = df_normalizado[df_normalizado["IEC.62443"] != ""].drop_duplicates()
    
    # 3. Agregação Semântica
    # Para cada norma única, agrupamos todas as descrições em um único bloco de texto.
    # Isso cria um 'âncora' semântica mais densa para o modelo de embeddings.
    dicionario_final = (
        df_normalizado.groupby("IEC.62443")["Name"]
        .unique()
        .apply(lambda x: " ".join(x))
        .to_dict()
    )
    
    # 4. Exportação para JSON
    nome_arquivo_saida = "../data/output/datasets/iec62443.json"
    with open(nome_arquivo_saida, "w", encoding="utf-8") as f:
        json.dump(dicionario_final, f, indent=4, ensure_ascii=False)
    
    print(f"Processamento concluído: {len(dicionario_final)} requisitos únicos mapeados em '{nome_arquivo_saida}'.")

# Executar a função no ambiente do notebook
gerar_dataset_iec62443_limpo()

Processamento concluído: 42 requisitos únicos mapeados em '../data/output/iec62443.json'.


## CWE

In [ ]:
import xml.etree.ElementTree as ET
import json
import re

def gerar_dataset_cwe_limpo():
    """
    Lê o XML oficial da MITRE, extrai os IDs e descrições das fraquezas,
    e exporta um JSON achatado otimizado para modelos de NLP.
    """
    print("Carregando e fazendo o parse do XML. Isso pode levar alguns segundos...")
    tree = ET.parse('../data/input/cwe/cwec_v4.19.1.xml')
    root = tree.getroot()
    
    # O XML do CWE utiliza Namespaces (ex: xmlns="http://cwe.mitre.org/cwe-7").
    # Precisamos capturar esse namespace dinamicamente para que a busca funcione.
    ns = {'cwe': root.tag.split('}')[0].strip('{')} if '}' in root.tag else {}
    
    # Definindo os caminhos de busca considerando o namespace
    busca_weaknesses = './/cwe:Weaknesses/cwe:Weakness' if ns else './/Weaknesses/Weakness'
    busca_desc = 'cwe:Description' if ns else 'Description'
    busca_ext_desc = 'cwe:Extended_Description' if ns else 'Extended_Description'

    dicionario_cwe = {}

    # Função auxiliar para extrair todo o texto, mesmo que haja tags HTML/XML aninhadas
    def extrair_texto_limpo(elemento):
        if elemento is None:
            return ""
        # itertext() pega todo o texto ignorando as tags internas (ex: <b>, <p>)
        texto = " ".join(elemento.itertext())
        # Limpa quebras de linha e múltiplos espaços
        texto = re.sub(r'\s+', ' ', texto).strip()
        return texto

    for weakness in root.findall(busca_weaknesses, ns):
        cwe_id = weakness.attrib.get('ID')
        name = weakness.attrib.get('Name', '')
        
        # Extrai a descrição curta e a estendida (rica em contexto de segurança)
        desc_elem = weakness.find(busca_desc, ns)
        description = extrair_texto_limpo(desc_elem)
        
        ext_desc_elem = weakness.find(busca_ext_desc, ns)
        ext_desc = extrair_texto_limpo(ext_desc_elem)
        
        if cwe_id:
            tag_cwe = f"CWE-{cwe_id}"
            
            # Concatena os campos para criar a "âncora semântica" do requisito
            texto_completo = f"{name}. {description} {ext_desc}".strip()
            dicionario_cwe[tag_cwe] = texto_completo

    # Exportação
    nome_arquivo_saida = "../data/output/datasets/cwe.json"
    with open(nome_arquivo_saida, 'w', encoding='utf-8') as f:
        json.dump(dicionario_cwe, f, indent=4, ensure_ascii=False)
        
    print(f"Processamento concluído: {len(dicionario_cwe)} CWEs mapeados e salvos em '{nome_arquivo_saida}'.")

# Executando a função com o arquivo anexado
gerar_dataset_cwe_limpo()

Carregando e fazendo o parse do XML. Isso pode levar alguns segundos...
Processamento concluído: 969 CWEs mapeados e salvos em '../data/output/cwe.json'.


## CIS Benchmark

In [ ]:
import requests
import yaml
import json
import re

def extrair_cis_docker_da_aqua(arquivo_saida):
    print("🌐 Baixando o CIS Docker Benchmark oficial da Aqua Security...")
    
    # URL direta para o arquivo YAML cru no GitHub deles (versão CIS 1.2.0)
    url_yaml = "https://raw.githubusercontent.com/aquasecurity/docker-bench/main/cfg/cis-1.6.0/definitions.yaml"
    
    try:
        resposta = requests.get(url_yaml)
        resposta.raise_for_status()
        # O BaseLoader evita que o YAML quebre com caracteres estranhos
        dados_cis = yaml.load(resposta.text, Loader=yaml.BaseLoader)
    except Exception as e:
        print(f"❌ Erro ao baixar o arquivo: {e}")
        return

    dataset_cis_docker = {}
    total_regras = 0

    print("🧩 Processando os grupos e extraindo as regras...")
    
    # O YAML é dividido em "groups" (Seções do CIS)
    grupos = dados_cis.get('groups', [])
    for grupo in grupos:
        grupo_id = str(grupo.get('id', ''))
        
        # Nós queremos APENAS a Seção 4 (Container Images and Build File)
        # Se quiser extrair o manual inteiro, basta remover o 'if' abaixo.
        if not grupo_id.startswith('4'):
            continue
            
        checks = grupo.get('checks', [])
        for check in checks:
            check_id = str(check.get('id', ''))
            descricao = check.get('description', '').strip()
            remediacao = check.get('remediation', '').strip()
            
            # Limpa o texto da remediação (remove quebras de linha e pipes)
            remediacao_limpa = re.sub(r'\s+', ' ', remediacao).strip()
            remediacao_limpa = remediacao_limpa.replace('|', '')
            
            # Remove a tag "(Scored) / (Not Scored)" do título para não sujar a IA
            descricao_limpa = re.sub(r'\s*\(Not Scored\)|\s*\(Scored\)', '', descricao, flags=re.IGNORECASE)
            
            # Concatena para criar o bloco denso de NLP
            texto_denso = f"{descricao_limpa}. {remediacao_limpa}".strip()
            
            if texto_denso:
                # Chave amigável, ex: "CIS_4.1"
                chave_nlp = f"CIS_{check_id}"
                dataset_cis_docker[chave_nlp] = texto_denso
                total_regras += 1

    # Exportação para o Dataset de Destino
    with open(arquivo_saida, 'w', encoding='utf-8') as f:
        json.dump(dataset_cis_docker, f, indent=4, ensure_ascii=False)

    print("="*40)
    print("🎯 RELATÓRIO DO DATASET CIS DOCKER")
    print("="*40)
    print(f"Regras da Seção 4 extraídas: {total_regras}")
    print(f"Arquivo salvo com sucesso em:  {arquivo_saida}")
    print("\n✅ Metade B (Destino) 100% Finalizada!")

# Execute o extrator
extrair_cis_docker_da_aqua('../data/output/datasets/cis_docker_benchmark.json')

🌐 Baixando o CIS Docker Benchmark oficial da Aqua Security...
🧩 Processando os grupos e extraindo as regras...
🎯 RELATÓRIO DO DATASET CIS DOCKER
Regras da Seção 4 extraídas: 12
Arquivo salvo com sucesso em:  ../data/output/cis_docker_benchmark.json

✅ Metade B (Destino) 100% Finalizada!


## Hadolint e ShellCheck

In [ ]:
import os
import subprocess
import json
import re

def limpar_texto_wiki_para_nlp(conteudo_md):
    # 1. Remove blocos de código (Sintaxe pura não ajuda a IA)
    texto = re.sub(r'```.*?```', '', conteudo_md, flags=re.DOTALL)
    
    # 2. Remove subtítulos "inúteis" para o modelo (Boilerplate)
    texto = re.sub(r'(?i)#*\s*(?:Problematic code|Correct code|Rationale|Exceptions)[:]*', '', texto)
    
    # 3. Remove apenas a tag da regra no topo (ex: "# DL4003") para não sujar o vocabulário
    texto = re.sub(r'^#+\s*(?:DL|SC)\d+\s*$', '', texto, flags=re.MULTILINE|re.IGNORECASE)
    
    # 4. Remove apenas os SÍMBOLOS '#' restantes, preservando o texto rico que estava ao lado deles
    texto = re.sub(r'#+', '', texto)
    
    # 5. Limpa a sintaxe de links Markdown [texto](url), mantendo o texto
    texto = re.sub(r'\[(.*?)\]\(.*?\)', r'\1', texto)
    
    # 6. Remove URLs soltas e os prefixos ("See also:", "Reference:", etc.)
    texto = re.sub(r'(?i)(?:see(?:\s+also)?|read(?:\s+more)?|reference[s]?|for\s+more\s+(?:info|details))?[\s:]*http[s]?://\S+', '', texto)
    
    # 7. Remove formatação pesada de Markdown (*, _, `, >, etc)
    texto = re.sub(r'[*`_~>|-]', '', texto)
    
    # 8. Achata o texto para gerar um bloco denso e contínuo
    texto_denso = re.sub(r'\s+', ' ', texto).strip()
    
    return texto_denso

def clonar_e_extrair_wikis(arquivo_saida):
    dataset_origem = {}
    total_dl = 0
    total_sc = 0

    print("="*55)
    print("🚀 INICIANDO INGESTÃO PADRONIZADA (WIKIS LOCAIS)")
    print("="*55)

    # Dicionário de configuração das Wikis
    repositorios_wiki = {
        "hadolint": {
            "url": "https://github.com/hadolint/hadolint.wiki.git",
            "pasta": "../data/input/hadolint/wiki_hadolint_temp",
            "regex": r'^DL\d{4}\.md$'
        },
        "shellcheck": {
            "url": "https://github.com/koalaman/shellcheck.wiki.git",
            "pasta": "../data/input/hadolint/wiki_shellcheck_temp",
            "regex": r'^SC\d{4}\.md$'
        }
    }

    # ---------------------------------------------------------
    # CLONAGEM E EXTRAÇÃO
    # ---------------------------------------------------------
    for projeto, config in repositorios_wiki.items():
        print(f"\n📚 Processando a Wiki do {projeto.upper()}...")
        pasta = config["pasta"]
        
        # Faz o Git Clone apenas se a pasta ainda não existir
        if not os.path.exists(pasta):
            print(f"   📥 Clonando repositório: {config['url']}")
            try:
                subprocess.run(["git", "clone", config["url"], pasta], check=True, capture_output=True)
            except subprocess.CalledProcessError as e:
                print(f"   ❌ Erro ao clonar a wiki do {projeto}. O Git retornou um erro.")
                continue
        else:
            print("   (Repositório já encontrado localmente. Lendo arquivos...)")

        # Varre a pasta da Wiki procurando os arquivos alvo
        for nome_arquivo in os.listdir(pasta):
            if re.match(config["regex"], nome_arquivo):
                caminho_completo = os.path.join(pasta, nome_arquivo)
                
                try:
                    with open(caminho_completo, 'r', encoding='utf-8') as f:
                        texto_limpo = limpar_texto_wiki_para_nlp(f.read())
                        
                    if texto_limpo:
                        id_regra = nome_arquivo.replace('.md', '')
                        dataset_origem[id_regra] = texto_limpo
                        
                        if projeto == "hadolint":
                            total_dl += 1
                        else:
                            total_sc += 1
                except Exception as e:
                    pass # Ignora silenciosamente se houver problema de leitura num arquivo específico

    # ---------------------------------------------------------
    # EXPORTAÇÃO
    # ---------------------------------------------------------
    with open(arquivo_saida, 'w', encoding='utf-8') as f:
        json.dump(dataset_origem, f, indent=4, ensure_ascii=False)

    print("\n" + "="*55)
    print("🎯 RELATÓRIO DO DATASET MESTRE DE ORIGEM")
    print("="*55)
    print(f"Regras Hadolint (DL) extraídas:   {total_dl}")
    print(f"Regras ShellCheck (SC) extraídas: {total_sc}")
    print(f"Total de vetores para a IA:       {total_dl + total_sc}")
    print(f"Arquivo gerado:                   {arquivo_saida}")
    print("\n✅ Base de Origem Consistente e 100% Finalizada!")

# Pode rodar! Ele criará as pastas wiki_hadolint_temp e wiki_shellcheck_temp no seu diretório.
clonar_e_extrair_wikis('../data/output/datasets/hadolint_rules.json')

🚀 INICIANDO INGESTÃO PADRONIZADA (WIKIS LOCAIS)

📚 Processando a Wiki do HADOLINT...
   📥 Clonando repositório: https://github.com/hadolint/hadolint.wiki.git

📚 Processando a Wiki do SHELLCHECK...
   📥 Clonando repositório: https://github.com/koalaman/shellcheck.wiki.git

🎯 RELATÓRIO DO DATASET MESTRE DE ORIGEM
Regras Hadolint (DL) extraídas:   69
Regras ShellCheck (SC) extraídas: 520
Total de vetores para a IA:       589
Arquivo gerado:                   ../data/output/hadolint_rules.json

✅ Base de Origem Consistente e 100% Finalizada!
